# ForceLorentzBalance — coupled low-Rm MHD `force_lorentz`

`MagneticCoupled` · ASN 28 validation form (TRUST-LiMe).



## Introduction

Validation of the **Lorentz-force coupling** between the magnetic induction
equation and the momentum equation of the same `Pb_MHD` problem. The problem
block contains **two equations**: `Convection_Diffusion_Magnetic_field` and the
optional `Navier_Stokes_standard`, coupled by the `force_lorentz` source term
attached to the momentum equation:

$$\mathbf{F} = \mathbf{J}\times\mathbf{b},\qquad \mathbf{J} = \frac{1}{\mu_0}\nabla\times\mathbf{b}$$

computed from the **lagged** magnetic field (explicit low-Rm coupling) and its
Green-Gauss element gradient. The source is expressed per unit mass, like every
TRUST momentum volumetric source (gravity/Boussinesq convention), and is
assembled on the face-based velocity unknown following the kernel
`Terme_Boussinesq_PolyMAC_CDO_Face` structure (face block + PolyMAC_MPFA
element block, Dirichlet-velocity faces skipped).

This notebook runs **one and only one test case**: `equilibre_lorentz`.

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | Lorentz hydrostatic balance | $p(y) = -y^2/(2\mu_0^2\rho) + C$ |
| 2 | curl-free force produces no flow | $\mathbf{u} \to 0$ |

With the exactly linear field $\mathbf{b} = (y, 0)$ ($\eta = 0$, no transport
velocity, so $\mathbf{b}$ is an exact equilibrium):

$$\nabla\times\mathbf{b} = -1 \;\Rightarrow\; \mathbf{J} = (0,0,-1)/\mu_0 \;\Rightarrow\; \mathbf{F} = \mathbf{J}\times\mathbf{b} = (0,-y,0)/\mu_0^2$$

$\mathbf{F}$ is a pure gradient, so the exact steady solution of the momentum
equation in the closed box is $\mathbf{u} = 0$ and
$p = -y^2/(2\mu_0^2\rho) + C$.  With $\mu_0 = \rho = 1$, the pressure
difference between the interior cell centres $(0.45, 0.45)$ and $(0.45, 0.75)$
is $p_2 - p_1 = -(0.75^2 - 0.45^2)/2 = -0.18$.

**Tolerances.** The Green-Gauss $\nabla\mathbf{b}$ is exact at interior cells
but carries the O(h) wall bias (homogeneous-Neumann boundary assumption), and
the discrete force balances the discrete pressure gradient only projectively:
the pressure difference converges to $-0.1886$ instead of $-0.18$ (~5 %, mesh
convergent) and the velocity transient decays exponentially without reaching
machine zero. The assertions below bound both effects.



In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()



## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.



In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case`."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

print("BUILD =", BUILD)



## Run

Sequential-only: the dataset is run by `trustutils` on one processor
(no `partition()`, no `nbProcs`).



In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "equilibre_lorentz.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("equilibre_lorentz_*.son"):
    shutil.copy2(son, build / son.name)



## Checks

### #1 — Lorentz hydrostatic balance (pressure difference)



In [ ]:
t1, p1 = read_son("equilibre_lorentz", "SONDE_P1")
t2, p2 = read_son("equilibre_lorentz", "SONDE_P2")
assert np.allclose(t1, t2), "probe time grids differ"

dp = p2 - p1
expected = -(0.75**2 - 0.45**2) / 2.0        # -0.18, with mu0 = rho = 1
print(f"p(0.45) = {p1[-1]:.6g}")
print(f"p(0.75) = {p2[-1]:.6g}")
print(f"p2 - p1 = {dp[-1]:.6g}   (analytic {expected:.4g}, "
      f"discrete balance ~ -0.1886 at h = 0.1)")
err = abs(dp[-1] - expected) / abs(expected)
print(f"relative error = {err:.3e}")
assert dp[-1] < 0, "wrong sign of the Lorentz pressure balance"
assert err < 0.10, f"pressure balance off: dp = {dp[-1]} (expected {expected} +/- 10 %)"
print("OK: Lorentz hydrostatic balance (sign and magnitude within 10 %).")



### #2 — curl-free force produces no flow



In [ ]:
def read_son_all(case, sonde):
    """Every entry (one per point/component) of probe `sonde` as (name, values)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

entries = read_son_all("equilibre_lorentz", "SONDE_U")
# The transient peaks at the first step (~3e-3) then decays like
# exp(-nu pi^2 t); the case early-stops on stationarity (t = 0.3).
# Check the relaxed state: |u| at the final time below 5e-4.
unorm = max(float(np.max(np.abs(v[-1:]))) for _, v in entries)
print(f"max |u| over all probe entries (final time): {unorm:.3e}")
for name, v in entries:
    print(f"  {name}: last |u| = {np.max(np.abs(v[-1:])):.3e}")
assert unorm < 5e-4, f"velocity did not relax: |u| = {unorm}"
print("OK: the curl-free Lorentz force is balanced by pressure, u -> 0.")



## Computer Performance


In [ ]:
run.tablePerf()

